<a href="https://colab.research.google.com/github/subudear/deep-learning/blob/main/assignment2/partA_method2/birdnet_finetune_original_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📝 Step 16: Summary and Next Steps

### Part A Results Summary

**Key Findings:**
1. ✅ Trained a BirdNet-inspired CNN on bird audio classification
2. ✅ Applied data augmentation (time stretch, pitch shift, noise, SpecAugment)
3. ✅ Handled class imbalance with weighted loss
4. ✅ Achieved validation accuracy: [See evaluation above]
5. ⚠️ Single-example classes likely have 0% accuracy (expected)

### What to Include in Your Report:

**Section 1: EDA**
- Sampling rate distribution
- Audio duration statistics
- Class imbalance analysis
- Background noise levels (SNR)

**Section 2: Preprocessing**
- Mel-spectrogram generation (parameters: n_mels, n_fft, hop_length)
- Standardization to 5 seconds
- Normalization approach

**Section 3: Data Augmentation**
- Time stretching (0.8-1.2x)
- Pitch shifting (±3 semitones)
- Gaussian noise addition
- Time/frequency masking (SpecAugment)
- Include spectrogram visualizations

**Section 4: Model Architecture**
- CNN structure (5 conv blocks)
- Number of parameters
- Feature extraction vs classification layers
- Diagram showing architecture flow

**Section 5: Training Strategy**
- 80/20 stratified split
- Single-example classes → validation
- Weighted CrossEntropy loss
- AdamW optimizer (LR=1e-4, WD=1e-4)
- ReduceLROnPlateau scheduler
- Early stopping (patience=7)

**Section 6: Results**
- Training curves (loss, accuracy, LR)
- Validation metrics (accuracy, F1-macro, F1-weighted)
- Confusion matrix for top classes
- Per-class performance analysis
- Single-example class performance

**Section 7: Discussion**
- What worked well
- What didn't work (single-example classes)
- How Part B with synthetic data might improve results
- Future improvements

### Next Steps for Part B:
1. Use AudioLDM2 to generate synthetic samples
2. Balance all classes (especially single-example ones)
3. Validate synthetic samples with BirdNet
4. Two approaches:
   - BirdNet embeddings + ML classifier
   - Finetune BirdNet on enriched dataset
5. Compare Part A vs Part B performance

### Hyperparameter Tuning Opportunities:
- Try different learning rates (1e-3, 5e-5)
- Experiment with batch sizes (16, 64)
- Adjust augmentation probabilities
- Try focal loss instead of weighted CE
- Freeze/unfreeze strategies (progressive unfreezing)
- Different model architectures (ResNet, EfficientNet)

---

**🎉 Part A Complete!**

Good luck with Part B and your report!

In [ ]:
# Save training history
import pickle

history_path = '/content/training_history.pkl'
with open(history_path, 'wb') as f:
    pickle.dump(history, f)
print(f"✓ Training history saved to {history_path}")

# Save label encoder
encoder_path = '/content/label_encoder.pkl'
with open(encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"✓ Label encoder saved to {encoder_path}")

# Copy model to Google Drive for persistence
import shutil

drive_model_path = '/content/drive/MyDrive/best_birdnet_model_partA.pth'
drive_history_path = '/content/drive/MyDrive/training_history_partA.pkl'
drive_encoder_path = '/content/drive/MyDrive/label_encoder_partA.pkl'

shutil.copy(SAVE_PATH, drive_model_path)
shutil.copy(history_path, drive_history_path)
shutil.copy(encoder_path, drive_encoder_path)

print(f"\n✓ All artifacts copied to Google Drive:")
print(f"  Model: {drive_model_path}")
print(f"  History: {drive_history_path}")
print(f"  Encoder: {drive_encoder_path}")

if os.path.exists(SUBMISSION_PATH):
    drive_submission_path = '/content/drive/MyDrive/submission_partA.csv'
    shutil.copy(SUBMISSION_PATH, drive_submission_path)
    print(f"  Submission: {drive_submission_path}")

## 💾 Step 15: Save Results and Model Artifacts

In [ ]:
# Test set prediction (update paths as needed)
TEST_AUDIO_DIR = '/content/test_audio'  # Update this!
TEST_CSV = '/content/drive/MyDrive/test.csv'  # Update this!
SUBMISSION_PATH = '/content/submission.csv'

# Check if test data exists
if os.path.exists(TEST_CSV):
    print("Test set found! Generating predictions...")

    # Load test metadata
    test_df = pd.read_csv(TEST_CSV)
    test_df['filepath'] = test_df[FILENAME_COLUMN].apply(lambda x: os.path.join(TEST_AUDIO_DIR, x))

    # Filter existing files
    test_df = test_df[test_df['filepath'].apply(os.path.exists)].reset_index(drop=True)
    print(f"Test samples: {len(test_df)}")

    # Create test dataset (no augmentation)
    class TestDataset(Dataset):
        def __init__(self, dataframe):
            self.dataframe = dataframe

        def __len__(self):
            return len(self.dataframe)

        def __getitem__(self, idx):
            filepath = self.dataframe.iloc[idx]['filepath']
            y, sr = load_and_preprocess_audio(filepath)
            mel_spec = audio_to_melspectrogram(y, sr)
            mel_spec = torch.FloatTensor(mel_spec).unsqueeze(0)
            return mel_spec

    test_dataset = TestDataset(test_df)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # Generate predictions
    model.eval()
    test_preds = []
    test_probs = []

    with torch.no_grad():
        for inputs in tqdm(test_loader, desc='Predicting on test set'):
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)

            test_preds.extend(predicted.cpu().numpy())
            test_probs.extend(probs.cpu().numpy())

    # Convert to class names
    test_pred_labels = label_encoder.inverse_transform(test_preds)

    # Create submission file
    submission_df = pd.DataFrame({
        FILENAME_COLUMN: test_df[FILENAME_COLUMN],
        LABEL_COLUMN: test_pred_labels
    })

    submission_df.to_csv(SUBMISSION_PATH, index=False)
    print(f"\n✓ Submission file saved to: {SUBMISSION_PATH}")
    print(f"  Predictions: {len(submission_df)}")
    print(f"\nFirst few predictions:")
    print(submission_df.head(10))
else:
    print(f"Test set not found at {TEST_CSV}")
    print("Skipping test predictions.")

## 🎯 Step 14: Generate Predictions for Test Set (if available)

If you have a test set, this cell will generate predictions for submission.

In [ ]:
# Analyze single-example classes performance
single_example_indices = [label_encoder.transform([cls])[0] for cls in single_example_classes]
single_mask = np.isin(all_labels, single_example_indices)

if single_mask.sum() > 0:
    single_labels = all_labels[single_mask]
    single_preds = all_preds[single_mask]
    single_accuracy = accuracy_score(single_labels, single_preds)

    print("\n" + "=" * 70)
    print("SINGLE-EXAMPLE CLASSES PERFORMANCE")
    print("=" * 70)
    print(f"Total single-example classes in validation: {len(single_example_classes)}")
    print(f"Samples: {single_mask.sum()}")
    print(f"Accuracy: {single_accuracy*100:.2f}%")
    print("\nAs expected, performance is poor for single-example classes.")
    print("Part B with synthetic data should improve this!")
    print("=" * 70)
else:
    print("\nNo single-example classes in validation set.")

In [ ]:
# Confusion Matrix (for subset of classes to keep it readable)
top_n_classes = 15
top_classes = [cls for cls, _ in sorted_classes[:top_n_classes]]
top_class_indices = [label_encoder.transform([cls])[0] for cls in top_classes]

# Filter predictions for top classes
mask = np.isin(all_labels, top_class_indices)
filtered_labels = all_labels[mask]
filtered_preds = all_preds[mask]

# Remap to 0-indexed for confusion matrix
label_mapping = {old_idx: new_idx for new_idx, old_idx in enumerate(top_class_indices)}
remapped_labels = np.array([label_mapping[l] for l in filtered_labels])
remapped_preds = np.array([label_mapping[p] for p in filtered_preds])

# Compute confusion matrix
cm = confusion_matrix(remapped_labels, remapped_preds)

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=top_classes, yticklabels=top_classes,
            cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix (Top {top_n_classes} Classes)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"\n✓ Confusion matrix plotted for top {top_n_classes} classes")

In [ ]:
# Calculate metrics
val_accuracy = accuracy_score(all_labels, all_preds)
val_f1_macro = f1_score(all_labels, all_preds, average='macro')
val_f1_weighted = f1_score(all_labels, all_preds, average='weighted')

print("=" * 70)
print("VALIDATION METRICS")
print("=" * 70)
print(f"Accuracy: {val_accuracy*100:.2f}%")
print(f"F1-Score (Macro): {val_f1_macro:.4f}")
print(f"F1-Score (Weighted): {val_f1_weighted:.4f}")
print("=" * 70)

# Per-class performance
print("\nPer-Class Performance (Top 20 classes by sample count):")
print("-" * 70)

# Get class-wise accuracy
class_correct = {}
class_total = {}

for true, pred in zip(all_labels, all_preds):
    class_name = label_encoder.classes_[true]
    if class_name not in class_total:
        class_total[class_name] = 0
        class_correct[class_name] = 0
    class_total[class_name] += 1
    if true == pred:
        class_correct[class_name] += 1

class_accuracies = {
    cls: (class_correct[cls] / class_total[cls] * 100) if class_total[cls] > 0 else 0
    for cls in class_total
}

# Sort by number of samples
sorted_classes = sorted(class_total.items(), key=lambda x: x[1], reverse=True)

for i, (cls, count) in enumerate(sorted_classes[:20]):
    acc = class_accuracies[cls]
    print(f"{i+1:2d}. {cls:30s} | Samples: {count:4d} | Accuracy: {acc:6.2f}%")

In [ ]:
# Load best model
checkpoint = torch.load(SAVE_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"  Validation Loss: {checkpoint['val_loss']:.4f}")
print(f"  Validation Accuracy: {checkpoint['val_acc']:.2f}%")

# Get predictions on validation set
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc='Evaluating'):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

print(f"\n✓ Predictions generated for {len(all_preds)} validation samples")

## 📈 Step 13: Load Best Model and Evaluate

Load the best checkpoint and perform detailed evaluation on validation set.

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

# Learning Rate
axes[2].plot(history['lr'], label='Learning Rate', marker='o', color='red')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

print("Training curves plotted successfully!")

## 📊 Step 12: Visualize Training History

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc='Training')
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc


def validate_epoch(model, loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc, all_preds, all_labels


# Training loop
print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print(f"This will take approximately 5-8 hours on GPU...")
print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")
print(f"Batches per epoch: Train={len(train_loader)}, Val={len(val_loader)}")
print("=" * 70)

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'lr': []
}

best_val_loss = float('inf')
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 70)

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

    # Validate
    val_loss, val_acc, val_preds, val_labels = validate_epoch(model, val_loader, criterion, device)

    # Learning rate scheduling
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    # Print epoch summary
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.2f}%")
    print(f"  Learning Rate: {current_lr:.2e}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, SAVE_PATH)
        print(f"  ✓ Best model saved! (Val Loss: {val_loss:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  No improvement for {epochs_without_improvement} epoch(s)")

    # Early stopping
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\n⚠️ Early stopping triggered after {epoch+1} epochs")
        break

print("\n" + "=" * 70)
print("TRAINING COMPLETE!")
print("=" * 70)
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Model saved to: {SAVE_PATH}")

## 🏋️ Step 11: Training Loop

**This cell will take the longest time to run (5-8 hours depending on GPU)**

Progress will be displayed with:
- Training/validation loss
- Training/validation accuracy
- Learning rate updates
- Best model checkpoints

In [ ]:
# Training hyperparameters
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 7
SAVE_PATH = '/content/best_birdnet_model.pth'

# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    verbose=True,
    min_lr=1e-7
)

print(f"Training Configuration:")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Weight Decay: {WEIGHT_DECAY}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Optimizer: AdamW")
print(f"  Scheduler: ReduceLROnPlateau")
print(f"  Loss: Weighted CrossEntropy")
print(f"\n✓ Training configuration ready!")

## ⚙️ Step 10: Training Configuration

Key hyperparameters for finetuning:
- **Learning Rate**: Start low (1e-4) to preserve pre-trained features
- **Loss Function**: Weighted CrossEntropy to handle class imbalance
- **Optimizer**: AdamW with weight decay
- **Scheduler**: ReduceLROnPlateau to adapt learning rate
- **Early Stopping**: Stop if no improvement for N epochs

In [ ]:
class BirdNetModel(nn.Module):
    """
    BirdNet-inspired CNN architecture for bird audio classification
    """

    def __init__(self, num_classes, input_channels=1, freeze_features=False):
        super(BirdNetModel, self).__init__()

        # Feature extraction layers (similar to BirdNet backbone)
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),

            # Block 5
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))  # Global average pooling
        )

        # Freeze feature extraction layers if finetuning
        if freeze_features:
            for param in self.features.parameters():
                param.requires_grad = False

        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

    def unfreeze_features(self):
        """Unfreeze feature layers for full finetuning"""
        for param in self.features.parameters():
            param.requires_grad = True


# Create model
model = BirdNetModel(num_classes=num_classes, freeze_features=False)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model created!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

## 🤖 Step 9: Define BirdNet Model

For this implementation, we'll create a CNN-based model inspired by BirdNet architecture.
BirdNet uses ResNet-like structure optimized for audio.

**Note:** For actual BirdNet pre-trained weights, you would load from the BirdNET-Analyzer package.
Here we'll create a similar architecture that can be trained or finetuned.

In [ ]:
# Create data loaders
BATCH_SIZE = 32  # Adjust based on GPU memory

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"\n✓ Datasets and DataLoaders created!")

In [ ]:
class BirdAudioDataset(Dataset):
    """
    PyTorch Dataset for bird audio classification
    """

    def __init__(self, dataframe, augment=False, augment_prob=0.5):
        self.dataframe = dataframe.reset_index(drop=True)
        self.augment = augment
        self.augment_prob = augment_prob
        self.aug = AudioAugmentation()

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        filepath = row['filepath']
        label = row['label_encoded']

        # Load and preprocess audio
        y, sr = load_and_preprocess_audio(filepath)

        # Apply augmentations (only during training)
        if self.augment and np.random.rand() < self.augment_prob:
            # Randomly apply augmentations
            if np.random.rand() < 0.3:
                y = self.aug.time_stretch(y)
                y = y[:MAX_LENGTH]  # Ensure correct length
                if len(y) < MAX_LENGTH:
                    y = np.pad(y, (0, MAX_LENGTH - len(y)), mode='constant')

            if np.random.rand() < 0.3:
                y = self.aug.pitch_shift(y, sr)

            if np.random.rand() < 0.3:
                y = self.aug.add_noise(y)

        # Convert to mel-spectrogram
        mel_spec = audio_to_melspectrogram(y, sr)

        # Apply spectrogram augmentations
        if self.augment and np.random.rand() < self.augment_prob:
            if np.random.rand() < 0.4:
                mel_spec = self.aug.time_mask(mel_spec)
            if np.random.rand() < 0.4:
                mel_spec = self.aug.freq_mask(mel_spec)

        # Convert to tensor and add channel dimension
        mel_spec = torch.FloatTensor(mel_spec).unsqueeze(0)  # (1, n_mels, time)
        label = torch.LongTensor([label])[0]

        return mel_spec, label


# Create datasets
train_dataset = BirdAudioDataset(train_df, augment=True, augment_prob=0.6)
val_dataset = BirdAudioDataset(val_df, augment=False)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

# Test dataset
sample_mel, sample_label = train_dataset[0]
print(f"\nSample mel-spectrogram shape: {sample_mel.shape}")
print(f"Sample label: {sample_label}")
print(f"Label name: {label_encoder.classes_[sample_label]}")

## 🗂️ Step 8: Create PyTorch Dataset and DataLoader

In [ ]:
class AudioAugmentation:
    """
    Audio augmentation techniques
    """

    @staticmethod
    def time_stretch(y, rate=None):
        """Stretch or compress audio in time"""
        if rate is None:
            rate = np.random.uniform(0.8, 1.2)
        return librosa.effects.time_stretch(y, rate=rate)

    @staticmethod
    def pitch_shift(y, sr, n_steps=None):
        """Shift pitch up or down"""
        if n_steps is None:
            n_steps = np.random.randint(-3, 4)  # -3 to +3 semitones
        return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

    @staticmethod
    def add_noise(y, noise_level=None):
        """Add Gaussian noise"""
        if noise_level is None:
            noise_level = np.random.uniform(0.001, 0.005)
        noise = np.random.randn(len(y)) * noise_level
        return y + noise

    @staticmethod
    def time_mask(mel_spec, max_mask_width=20):
        """Mask random time segments (SpecAugment)"""
        mel_spec = mel_spec.copy()
        width = np.random.randint(1, max_mask_width)
        start = np.random.randint(0, mel_spec.shape[1] - width)
        mel_spec[:, start:start+width] = 0
        return mel_spec

    @staticmethod
    def freq_mask(mel_spec, max_mask_height=20):
        """Mask random frequency bands (SpecAugment)"""
        mel_spec = mel_spec.copy()
        height = np.random.randint(1, max_mask_height)
        start = np.random.randint(0, mel_spec.shape[0] - height)
        mel_spec[start:start+height, :] = 0
        return mel_spec


# Test augmentations
y_test, sr_test = load_and_preprocess_audio(test_filepath)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))

# Original
mel_orig = audio_to_melspectrogram(y_test, sr_test)
librosa.display.specshow(mel_orig, ax=axes[0, 0], x_axis='time', y_axis='mel')
axes[0, 0].set_title('Original')

# Time stretch
y_stretched = AudioAugmentation.time_stretch(y_test, rate=1.1)
y_stretched = y_stretched[:len(y_test)]  # Ensure same length
mel_stretched = audio_to_melspectrogram(y_stretched, sr_test)
librosa.display.specshow(mel_stretched, ax=axes[0, 1], x_axis='time', y_axis='mel')
axes[0, 1].set_title('Time Stretched (1.1x)')

# Pitch shift
y_pitched = AudioAugmentation.pitch_shift(y_test, sr_test, n_steps=2)
mel_pitched = audio_to_melspectrogram(y_pitched, sr_test)
librosa.display.specshow(mel_pitched, ax=axes[0, 2], x_axis='time', y_axis='mel')
axes[0, 2].set_title('Pitch Shifted (+2 semitones)')

# Add noise
y_noisy = AudioAugmentation.add_noise(y_test, noise_level=0.003)
mel_noisy = audio_to_melspectrogram(y_noisy, sr_test)
librosa.display.specshow(mel_noisy, ax=axes[1, 0], x_axis='time', y_axis='mel')
axes[1, 0].set_title('With Background Noise')

# Time mask
mel_time_masked = AudioAugmentation.time_mask(mel_orig.copy(), max_mask_width=30)
librosa.display.specshow(mel_time_masked, ax=axes[1, 1], x_axis='time', y_axis='mel')
axes[1, 1].set_title('Time Masked')

# Freq mask
mel_freq_masked = AudioAugmentation.freq_mask(mel_orig.copy(), max_mask_height=20)
librosa.display.specshow(mel_freq_masked, ax=axes[1, 2], x_axis='time', y_axis='mel')
axes[1, 2].set_title('Frequency Masked')

plt.tight_layout()
plt.show()

print("✓ Augmentation functions ready!")

## 🎨 Step 7: Data Augmentation

Augmentation techniques for audio:
1. **Time Stretching**: Speed up/slow down without changing pitch
2. **Pitch Shifting**: Change pitch without changing speed
3. **Background Noise**: Add Gaussian noise
4. **Time/Frequency Masking**: SpecAugment technique
5. **Mixup**: Blend two samples (optional, advanced)

In [ ]:
def load_and_preprocess_audio(filepath, sr=SAMPLE_RATE, duration=DURATION):
    """
    Load audio file and convert to standardized format
    """
    # Load audio
    y, orig_sr = librosa.load(filepath, sr=sr)

    # Standardize length (pad or crop to fixed duration)
    target_length = sr * duration

    if len(y) < target_length:
        # Pad with zeros
        y = np.pad(y, (0, target_length - len(y)), mode='constant')
    else:
        # Random crop during training, center crop during validation
        start = np.random.randint(0, len(y) - target_length + 1)
        y = y[start:start + target_length]

    return y, sr


def audio_to_melspectrogram(y, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """
    Convert audio waveform to mel-spectrogram
    """
    # Compute mel-spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=n_mels,
        n_fft=n_fft,
        hop_length=hop_length
    )

    # Convert to log scale (dB)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    # Normalize to [0, 1]
    mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)

    return mel_spec_db


# Test preprocessing on a sample
test_filepath = train_df.iloc[0]['filepath']
test_label = train_df.iloc[0][LABEL_COLUMN]

y, sr = load_and_preprocess_audio(test_filepath)
mel_spec = audio_to_melspectrogram(y, sr)

print(f"\nTest preprocessing:")
print(f"  Audio shape: {y.shape}")
print(f"  Mel-spectrogram shape: {mel_spec.shape}")
print(f"  Label: {test_label}")

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Waveform
librosa.display.waveshow(y, sr=sr, ax=axes[0])
axes[0].set_title(f'Waveform: {test_label}')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

# Mel-spectrogram
img = librosa.display.specshow(mel_spec, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel', ax=axes[1])
axes[1].set_title(f'Mel-Spectrogram: {test_label}')
fig.colorbar(img, ax=axes[1], format='%+2.0f')

plt.tight_layout()
plt.show()

print("\n✓ Preprocessing functions ready!")

In [ ]:
# Audio preprocessing configuration
SAMPLE_RATE = 32000  # BirdNet typically uses 48kHz, but 32kHz is good for balance
DURATION = 5  # seconds - standardize all clips to 5 seconds
N_MELS = 128  # number of mel bands
N_FFT = 2048  # FFT window size
HOP_LENGTH = 512  # overlap between windows
MAX_LENGTH = SAMPLE_RATE * DURATION

print(f"Audio Configuration:")
print(f"  Sample Rate: {SAMPLE_RATE} Hz")
print(f"  Duration: {DURATION} seconds")
print(f"  Mel Bands: {N_MELS}")
print(f"  FFT Size: {N_FFT}")
print(f"  Hop Length: {HOP_LENGTH}")
print(f"  Max Samples: {MAX_LENGTH}")

## 🎵 Step 6: Audio Preprocessing & Feature Extraction

We'll use **mel-spectrograms** as input to BirdNet:
- Convert audio waveform → frequency representation
- Mel-scale mimics bird auditory perception
- Results in 2D image-like representation suitable for CNNs

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(df[LABEL_COLUMN])

train_df['label_encoded'] = label_encoder.transform(train_df[LABEL_COLUMN])
val_df['label_encoded'] = label_encoder.transform(val_df[LABEL_COLUMN])

num_classes = len(label_encoder.classes_)
print(f"Number of classes: {num_classes}")
print(f"Class names (first 10): {label_encoder.classes_[:10]}")

# Calculate class weights for handling imbalance
class_weights = []
for i in range(num_classes):
    class_name = label_encoder.classes_[i]
    count = (train_df[LABEL_COLUMN] == class_name).sum()
    weight = 1.0 / count if count > 0 else 0.0
    class_weights.append(weight)

class_weights = torch.FloatTensor(class_weights)
class_weights = class_weights / class_weights.sum() * num_classes  # Normalize
print(f"\nClass weights computed (will be used in loss function)")

In [ ]:
# Prepare dataframe with full file paths
df['filepath'] = df[FILENAME_COLUMN].apply(lambda x: os.path.join(EXTRACT_TO, x))

# Verify files exist
existing_files = df['filepath'].apply(os.path.exists)
print(f"Files found: {existing_files.sum()} / {len(df)}")
df = df[existing_files].reset_index(drop=True)
print(f"Dataset after filtering: {len(df)} samples")

# Identify single vs multi-example classes
class_counts = df[LABEL_COLUMN].value_counts()
single_example_classes = class_counts[class_counts == 1].index.tolist()
multi_example_classes = class_counts[class_counts > 1].index.tolist()

# Split dataframe
df_single = df[df[LABEL_COLUMN].isin(single_example_classes)]
df_multi = df[df[LABEL_COLUMN].isin(multi_example_classes)]

print(f"\nSingle-example samples: {len(df_single)}")
print(f"Multi-example samples: {len(df_multi)}")

# Stratified split for multi-example classes
train_multi, val_multi = train_test_split(
    df_multi,
    test_size=0.2,
    stratify=df_multi[LABEL_COLUMN],
    random_state=42
)

# Single-example classes go to validation
val_single = df_single

# Combine
train_df = train_multi.reset_index(drop=True)
val_df = pd.concat([val_multi, val_single]).reset_index(drop=True)

print(f"\n{'='*60}")
print(f"TRAIN/VALIDATION SPLIT RESULTS")
print(f"{'='*60}")
print(f"Training samples: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation samples: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"\nUnique species in train: {train_df[LABEL_COLUMN].nunique()}")
print(f"Unique species in val: {val_df[LABEL_COLUMN].nunique()}")
print(f"Single-example classes in val: {len(val_single)}")
print(f"\n✓ Split complete!")

## ✂️ Step 5: Train/Validation Split (80/20 Stratified)

**Strategy:**
- Multi-example classes: Stratified split to maintain class proportions
- Single-example classes: All go to validation (expected to fail in Part A, but will be used in Part B with synthetic data)

In [ ]:
# Display audio analysis results
print("=" * 60)
print("AUDIO PROPERTIES ANALYSIS")
print("=" * 60)

print("\n1. SAMPLING RATE DISTRIBUTION:")
print(df_audio['sampling_rate'].value_counts())
print(f"\nMost common sampling rate: {df_audio['sampling_rate'].mode()[0]} Hz")

print("\n2. DURATION STATISTICS:")
print(df_audio['duration'].describe())

print("\n3. BACKGROUND NOISE LEVELS:")
print(df_audio[['min_rms', 'mean_rms', 'mean_zcr', 'snr_db']].describe())

# Visualizations
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Sampling rate
axes[0, 0].hist(df_audio['sampling_rate'], bins=20, edgecolor='black')
axes[0, 0].set_title('Sampling Rate Distribution')
axes[0, 0].set_xlabel('Sampling Rate (Hz)')
axes[0, 0].set_ylabel('Count')

# Duration
axes[0, 1].hist(df_audio['duration'], bins=30, edgecolor='black')
axes[0, 1].set_title('Audio Duration Distribution')
axes[0, 1].set_xlabel('Duration (seconds)')
axes[0, 1].set_ylabel('Count')

# RMS Energy
axes[0, 2].hist(df_audio['min_rms'], bins=30, edgecolor='black', color='orange')
axes[0, 2].set_title('Background Noise Floor (Min RMS)')
axes[0, 2].set_xlabel('Min RMS Energy')
axes[0, 2].set_ylabel('Count')

# Mean RMS
axes[1, 0].hist(df_audio['mean_rms'], bins=30, edgecolor='black', color='green')
axes[1, 0].set_title('Mean RMS Energy')
axes[1, 0].set_xlabel('Mean RMS')
axes[1, 0].set_ylabel('Count')

# Zero Crossing Rate
axes[1, 1].hist(df_audio['mean_zcr'], bins=30, edgecolor='black', color='purple')
axes[1, 1].set_title('Zero Crossing Rate')
axes[1, 1].set_xlabel('Mean ZCR')
axes[1, 1].set_ylabel('Count')

# SNR
axes[1, 2].hist(df_audio['snr_db'].dropna(), bins=30, edgecolor='black', color='red')
axes[1, 2].set_title('Signal-to-Noise Ratio')
axes[1, 2].set_xlabel('SNR (dB)')
axes[1, 2].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("\n✓ EDA Complete!")

In [ ]:
# Audio properties analysis (sampling rate, duration, noise)
def analyze_audio_properties(df, audio_dir, num_samples=200):
    """
    Analyze sampling rates, durations, and background noise levels
    """
    results = []

    # Sample random files for analysis
    sample_df = df.sample(min(num_samples, len(df)), random_state=42)

    for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Analyzing audio"):
        filename = row[FILENAME_COLUMN]
        filepath = os.path.join(audio_dir, filename)

        if not os.path.exists(filepath):
            continue

        try:
            # Load audio (sr=None preserves original sampling rate)
            y, sr = librosa.load(filepath, sr=None)
            duration = librosa.get_duration(y=y, sr=sr)

            # RMS energy for noise floor estimation
            rms = librosa.feature.rms(y=y)[0]
            min_rms = np.min(rms)
            mean_rms = np.mean(rms)

            # Zero-crossing rate
            zcr = librosa.feature.zero_crossing_rate(y)[0]
            mean_zcr = np.mean(zcr)

            # Estimate SNR (assuming first/last 0.5s is background noise)
            noise_duration = int(0.5 * sr)
            if len(y) > 2 * noise_duration:
                noise = np.concatenate([y[:noise_duration], y[-noise_duration:]])
                noise_power = np.mean(noise ** 2)
                signal_power = np.mean(y ** 2)
                snr_db = 10 * np.log10(signal_power / noise_power) if noise_power > 0 else float('inf')
            else:
                snr_db = np.nan

            results.append({
                'filename': filename,
                'sampling_rate': sr,
                'duration': duration,
                'num_samples': len(y),
                'min_rms': min_rms,
                'mean_rms': mean_rms,
                'mean_zcr': mean_zcr,
                'snr_db': snr_db
            })
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    return pd.DataFrame(results)

# Run analysis
print("Analyzing audio properties (this may take a few minutes)...")
df_audio = analyze_audio_properties(df, EXTRACT_TO, num_samples=200)

In [ ]:
# Visualize class distribution
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
class_counts.head(20).plot(kind='bar')
plt.title('Top 20 Classes by Sample Count')
plt.xlabel('Species')
plt.ylabel('Number of Samples')
plt.xticks(rotation=45, ha='right')

plt.subplot(1, 2, 2)
plt.hist(class_counts, bins=50, edgecolor='black')
plt.title('Distribution of Samples per Class')
plt.xlabel('Number of Samples')
plt.ylabel('Number of Classes')
plt.yscale('log')

plt.tight_layout()
plt.show()

print(f"\n⚠️ Class imbalance detected!")
print(f"Max samples: {class_counts.max()}, Min samples: {class_counts.min()}")
print(f"This will require: weighted loss OR focal loss OR oversampling")

In [ ]:
# Class distribution analysis
class_counts = df[LABEL_COLUMN].value_counts()

print(f"Number of unique species: {len(class_counts)}")
print(f"\nClass distribution statistics:")
print(class_counts.describe())

print(f"\nClasses with only 1 example: {(class_counts == 1).sum()}")
print(f"Classes with only 2-5 examples: {((class_counts >= 2) & (class_counts <= 5)).sum()}")
print(f"Classes with >100 examples: {(class_counts > 100).sum()}")

# Identify single-example classes
single_example_classes = class_counts[class_counts == 1].index.tolist()
print(f"\nSingle-example classes ({len(single_example_classes)}):")
print(single_example_classes[:10] if len(single_example_classes) > 10 else single_example_classes)

In [ ]:
# Load metadata
# Adjust column names based on your actual CSV structure
# Common structures: filename, species, label, primary_label, etc.

df = pd.read_csv(METADATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")

# Identify the label column (update this based on your CSV)
# Common names: 'species', 'label', 'primary_label', 'class'
LABEL_COLUMN = 'primary_label'  # UPDATE THIS based on your CSV!
FILENAME_COLUMN = 'filename'     # UPDATE THIS based on your CSV!

# Check if columns exist
if LABEL_COLUMN not in df.columns:
    print(f"\n⚠️ Warning: '{LABEL_COLUMN}' not found in columns. Available columns: {df.columns.tolist()}")
    print("Please update LABEL_COLUMN in the cell above.")
else:
    print(f"\n✓ Using '{LABEL_COLUMN}' as label column")
    print(f"✓ Using '{FILENAME_COLUMN}' as filename column")

## 📊 Step 4: Load Metadata and Perform EDA

Understanding the dataset is crucial before preprocessing.

In [ ]:
# Core libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Audio processing
import librosa
import librosa.display
import soundfile as sf

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T

# Set random seeds for reproducibility
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 📚 Step 3: Import Libraries

In [ ]:
# Install required packages
!pip install -q librosa soundfile audioread torchaudio timm scikit-learn

# Install BirdNET-Analyzer (contains pre-trained BirdNet model)
!pip install -q git+https://github.com/kahst/BirdNET-Analyzer.git

print("✓ All dependencies installed")

## 📦 Step 2: Install Dependencies

Installing required libraries for audio processing and deep learning.

In [ ]:
# Extract train_audio.zip to Colab runtime (faster I/O)
# Update the path below to match your Google Drive location

ZIP_PATH = '/content/drive/MyDrive/train_audio.zip'  # Update this path!
EXTRACT_TO = '/content/train_audio'
METADATA_PATH = '/content/drive/MyDrive/train.csv'  # Update if different

# Extract zip file
import zipfile
from tqdm import tqdm

print(f"Extracting {ZIP_PATH} to {EXTRACT_TO}...")
print("This may take 5-10 minutes for 16GB...")

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    members = zip_ref.namelist()
    for member in tqdm(members, desc="Extracting"):
        zip_ref.extract(member, EXTRACT_TO)

print(f"\n✓ Extracted {len(members)} files to {EXTRACT_TO}")

In [ ]:
# Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print("✓ Google Drive mounted successfully")

## 📋 Step 1: Mount Google Drive and Extract Data

**Note:** Data will be extracted to Colab's local runtime for faster I/O (10-100x faster than Drive)

# 🐦 Part A: Bird Audio Classification - BirdNet Finetuning

## Project Overview
This notebook implements **Part A** of the bird audio classification competition:
- **Goal:** Finetune BirdNet for best possible predictions on validation data
- **Data:** Original audio samples (no synthetic augmentation)
- **Split:** 80% train, 20% validation (stratified)
- **Approach:** Transfer learning with BirdNet pre-trained weights

## Pipeline Steps:
1. ✅ Mount Google Drive & Extract Data
2. 📊 Exploratory Data Analysis (EDA)
3. 🔧 Audio Preprocessing (Mel-Spectrograms)
4. ✂️ Train/Validation Split (handle single-example classes)
5. 🎨 Data Augmentation
6. 🤖 Load & Modify BirdNet
7. 🏋️ Training with Monitoring
8. 📈 Evaluation & Metrics
9. 🎯 Predictions

**Estimated Runtime:** 6-10 hours total (including training)